In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import holidays
from pygam import LinearGAM, s, f
from datetime import timedelta
from pathlib import Path
from tqdm import tqdm  
from rapidfuzz import process, fuzz
import os 
from quantile_forest import RandomForestQuantileRegressor


In [ ]:
#get node we are on
print(os.uname())

In [ ]:
#preprocessing US data
#load EIA 930 via PUDL
import pyarrow.parquet as pq
import pyarrow.fs as pafs

s3, path = pafs.FileSystem.from_uri("s3://pudl.catalyst.coop/nightly/core_eia930__hourly_operations.parquet")
s3 = pafs.S3FileSystem(anonymous=True, region="us-west-2")

df_y = pq.read_table(
    "pudl.catalyst.coop/nightly/core_eia930__hourly_operations.parquet",
    filesystem=s3
).to_pandas(types_mapper=pd.ArrowDtype)

print(df_y.columns)

In [ ]:
#lagged load variables: load_24, load_168
df_y['load_24'] = (
    df_y.groupby('balancing_authority_code_eia')
            ['demand_reported_mwh']
            .shift(24)
)

df_y['load_168'] = (
    df_y.groupby('balancing_authority_code_eia')
            ['demand_reported_mwh']
            .shift(168)
)

#target variable: load
y=df_y["demand_reported_mwh"]

#print columns for all features
print(df_y.columns)


In [ ]:
#add holiday data
us_holidays = holidays.US()
#add calender variables: Time of Day, Day of Week, Time of Year, Holiday
df_y["hour"] = df_y["datetime_utc"].dt.hour
df_y["dayofweek"] = df_y["datetime_utc"].dt.dayofweek
df_y["dayofyear"] = df_y["datetime_utc"].dt.dayofyear
df_y["is_holiday"] = df_y["datetime_utc"].dt.date.apply(lambda x: x in us_holidays)
df_y["is_holiday"] = df_y["is_holiday"].astype(int)

#print columns for all features
print(df_y.columns)
print(df_y.head())
#count how many unique BAs we have
print(df_y['balancing_authority_code_eia'].nunique())

In [ ]:
# Load and concat CONSUSII temperature files 
df_temp = pd.concat([
    pd.read_csv(f"BA_Load_Temp_Merge_{yr}.csv")
    for yr in range(2016, 2023)
], ignore_index=True)

df_temp["Time"] = pd.to_datetime(df_temp["Time"], utc=True)

print(df_temp.columns)
print(df_temp.head())
#unique BAs in temp data
print(df_temp['BA_NAME'].nunique())

In [ ]:
# Add smoothed temperature variables 
# Exponential smoothing with different decay rates (following base paper)
# alpha_95: slow decay (captures heat accumulation effect)
# alpha_99: very slow decay (captures prolonged heat stress effect)

df_temp = df_temp.sort_values(["ID", "Time"]).reset_index(drop=True)

alpha_95 = 0.05
alpha_99 = 0.01

df_temp["temp_s95"] = (
    df_temp.groupby("ID")["avg_temp_K"]
    .transform(lambda x: x.ewm(alpha=alpha_95, adjust=False).mean())
)
df_temp["temp_s99"] = (
    df_temp.groupby("ID")["avg_temp_K"]
    .transform(lambda x: x.ewm(alpha=alpha_99, adjust=False).mean())
)

print(df_temp[["Time", "ID", "avg_temp_K", "temp_s95", "temp_s99"]].head(10))


In [ ]:
#  5. BA crosswalk: CONSUSII BA_NAME to EIA code 
eia_ba_names = {
    "AEC":  "POWERSOUTH ENERGY COOPERATIVE",
    "AECI": "ASSOCIATED ELECTRIC COOPERATIVE INC",
    "AVA":  "AVISTA CORPORATION",
    "AZPS": "ARIZONA PUBLIC SERVICE COMPANY",
    "BANC": "BALANCING AUTHORITY OF NORTHERN CALIFORNIA",
    "BPAT": "BONNEVILLE POWER ADMINISTRATION",
    "CHPD": "PUBLIC UTILITY DISTRICT NO 1 OF CHELAN COUNTY",
    "CISO": "CALIFORNIA INDEPENDENT SYSTEM OPERATOR",
    "CPLE": "DUKE ENERGY PROGRESS EAST",
    "CPLW": "DUKE ENERGY PROGRESS WEST",
    "DUK":  "DUKE ENERGY CAROLINAS",
    "EPE":  "EL PASO ELECTRIC COMPANY",
    "ERCO": "ELECTRIC RELIABILITY COUNCIL OF TEXAS",
    "FMPP": "FLORIDA MUNICIPAL POWER POOL",
    "FPC":  "DUKE ENERGY FLORIDA",
    "FPL":  "FLORIDA POWER AND LIGHT COMPANY",
    "GVL":  "GAINESVILLE REGIONAL UTILITIES",
    "HST":  "CITY OF HOMESTEAD",
    "IID":  "IMPERIAL IRRIGATION DISTRICT",
    "IPCO": "IDAHO POWER COMPANY",
    "ISNE": "ISO NEW ENGLAND",
    "JEA":  "JEA",
    "LDWP": "LOS ANGELES DEPARTMENT OF WATER AND POWER",
    "LGEE": "LOUISVILLE GAS AND ELECTRIC COMPANY KENTUCKY UTILITIES COMPANY",
    "MISO": "MIDCONTINENT INDEPENDENT SYSTEM OPERATOR",
    "NEVP": "NEVADA POWER COMPANY",
    "NWMT": "NORTHWESTERN CORPORATION",
    "NYIS": "NEW YORK INDEPENDENT SYSTEM OPERATOR",
    "OVEC": "OHIO VALLEY ELECTRIC CORPORATION",
    "PACE": "PACIFICORP EAST",
    "PACW": "PACIFICORP WEST",
    "PGE":  "PORTLAND GENERAL ELECTRIC COMPANY",
    "PJM":  "PJM INTERCONNECTION",
    "PNM":  "PUBLIC SERVICE COMPANY OF NEW MEXICO",
    "PSCO": "PUBLIC SERVICE COMPANY OF COLORADO",
    "PSEI": "PUGET SOUND ENERGY",
    "SC":   "SOUTH CAROLINA ELECTRIC GAS COMPANY",
    "SCL":  "SEATTLE CITY LIGHT",
    "SEC":  "SEMINOLE ELECTRIC COOPERATIVE INC",
    "SEPA": "SOUTHEASTERN POWER ADMINISTRATION",
    "SOCO": "SOUTHERN COMPANY SERVICES INC",
    "SPA":  "SOUTHWESTERN POWER ADMINISTRATION",
    "SRP":  "SALT RIVER PROJECT",
    "SWPP": "SOUTHWEST POWER POOL",
    "TAL":  "CITY OF TALLAHASSEE",
    "TEC":  "TAMPA ELECTRIC COMPANY",
    "TEPC": "TUCSON ELECTRIC POWER COMPANY",
    "TIDC": "TURLOCK IRRIGATION DISTRICT",
    "TPWR": "CITY OF TACOMA",
    "TVA":  "TENNESSEE VALLEY AUTHORITY",
    "WACM": "WESTERN AREA POWER ADMINISTRATION ROCKY MOUNTAIN REGION",
    "WALC": "WESTERN AREA POWER ADMINISTRATION DESERT SOUTHWEST REGION",
    "WAUW": "WESTERN AREA POWER ADMINISTRATION UPPER GREAT PLAINS WEST",
    "GWA":  "NATURENER GLACIER WIND",
    "WWA":  "NATURENER WIND WATCH",
    "YAD":  "ALCOA POWER GENERATING INC YADKIN DIVISION",
    "DOPD": "PUBLIC UTILITY DISTRICT NO 1 OF DOUGLAS COUNTY",
    "GCPD": "PUBLIC UTILITY DISTRICT NO 2 OF GRANT COUNTY",
    "NSB":  "UTILITIES COMMISSION OF NEW SMYRNA BEACH",
}

In [ ]:
#  Build crosswalk and merge 
def match_ba(name, lookup, threshold=80):
    result = process.extractOne(name.upper(), lookup.values(), scorer=fuzz.token_sort_ratio)
    if result and result[1] >= threshold:
        return next(k for k, v in lookup.items() if v == result[0])
    return None

# Build crosswalk
consus_bas = df_temp[["ID", "BA_NAME"]].drop_duplicates()
consus_bas["ID"] = consus_bas["ID"].astype(int)
consus_bas["balancing_authority_code_eia"] = consus_bas["BA_NAME"].apply(
    lambda x: match_ba(x, eia_ba_names)
)

manual_fixes = {
    5326:  "DOPD",
    5748:  "EEI",
    12825: "NWMT",
    17539: "SCEG",   
    17543: "SC",     
    18429: "TPWR",
    55372: "HGMA",
    56090: "GRIF",
    56365: "GWA",
    58790: "GRID",
}

consus_bas["balancing_authority_code_eia"] = consus_bas.apply(
    lambda row: manual_fixes.get(row["ID"], row["balancing_authority_code_eia"]),
    axis=1
)

# Build a simple ID to code dict and map directly onto df_temp (avoids merge collision)
id_to_code = consus_bas.set_index("ID")["balancing_authority_code_eia"].to_dict()
df_temp["balancing_authority_code_eia"] = df_temp["ID"].astype(int).map(id_to_code)

# Fix datetime dtypes
df_y["datetime_utc"] = df_y["datetime_utc"].astype("datetime64[us, UTC]")
df_temp["Time"] = pd.to_datetime(df_temp["Time"], utc=True).astype("datetime64[us, UTC]")

# Merge temp into EIA-930 
df = df_y.merge(
    df_temp[["balancing_authority_code_eia", "Time", "avg_temp_K", "temp_s95", "temp_s99"]],
    left_on=["balancing_authority_code_eia", "datetime_utc"],
    right_on=["balancing_authority_code_eia", "Time"],
    how="left"
).drop(columns="Time")

#Keep only needed columns 
keep = [
    "balancing_authority_code_eia",
    "datetime_utc",
    "demand_adjusted_mwh",
    "load_24",
    "load_168",
    "hour",
    "dayofweek",
    "dayofyear",
    "is_holiday",
    "avg_temp_K",
    "temp_s95",   
    "temp_s99",   
]
df = df[keep]

df = df[
    (df["datetime_utc"] >= "2016-07-01") &
    (df["datetime_utc"] < "2023-01-01")
]

print(df.shape)
print(f"Rows with temperature: {df['avg_temp_K'].notna().mean():.1%}")


print(df.shape)
print(f"Rows with temperature: {df['avg_temp_K'].notna().mean():.1%}")
print(df.head())



In [ ]:
ba_coverage = df.groupby("balancing_authority_code_eia")["avg_temp_K"].apply(
    lambda x: x.notna().mean()
).sort_values()
print(ba_coverage[ba_coverage < 1.0])


In [ ]:
# Find BAs with 0% temp coverage and drop them entirely
ba_temp_coverage = df.groupby("balancing_authority_code_eia")["avg_temp_K"].apply(
    lambda x: x.notna().mean()
)
zero_coverage_bas = ba_temp_coverage[ba_temp_coverage == 0].index.tolist()
print("Dropping BAs with 0% temp coverage:", zero_coverage_bas)

df = df[~df["balancing_authority_code_eia"].isin(zero_coverage_bas)]

# For BAs with partial coverage, forward-fill then backward-fill gaps
df = df.sort_values(["balancing_authority_code_eia", "datetime_utc"])
df["avg_temp_K"] = df.groupby("balancing_authority_code_eia")["avg_temp_K"].transform(
    lambda x: x.ffill().bfill()
)

# Drop remaining NaNs in load lags
df = df.dropna(subset=["load_24", "load_168"])

print(df.shape)
print(f"BAs remaining: {df['balancing_authority_code_eia'].nunique()}")
print(f"Rows with temperature: {df['avg_temp_K'].notna().mean():.1%}")


In [ ]:
date_2020 = pd.to_datetime("2020-01-01", utc=True)
date_2021 = pd.to_datetime("2021-01-01", utc=True)

df_train = df[df["datetime_utc"] < date_2020]
df_val   = df[(df["datetime_utc"] >= date_2020) & (df["datetime_utc"] < date_2021)]
df_test  = df[df["datetime_utc"] >= date_2021]

drop_cols = ["demand_adjusted_mwh", "load_24", "load_168", "avg_temp_K"]

for name, split in [("train", df_train), ("val", df_val), ("test", df_test)]:
    split = split.replace([np.inf, -np.inf], np.nan).dropna(subset=drop_cols)
    print(f"{name}: {split.shape}")

df_train = df_train.replace([np.inf, -np.inf], np.nan).dropna(subset=drop_cols)
df_val   = df_val.replace([np.inf, -np.inf], np.nan).dropna(subset=drop_cols)
df_test  = df_test.replace([np.inf, -np.inf], np.nan).dropna(subset=drop_cols)

features = ["hour", "dayofweek", "dayofyear", "is_holiday", "avg_temp_K", "load_24", "load_168"]

X_train, y_train = df_train[features], df_train["demand_adjusted_mwh"]
X_val,   y_val   = df_val[features],   df_val["demand_adjusted_mwh"]
X_test,  y_test  = df_test[features],  df_test["demand_adjusted_mwh"]

print(X_train.shape, X_val.shape, X_test.shape)


**Fit GAM**

In [ ]:
from pygam import LinearGAM, s, f, te

def fit_gam_per_ba(df_train):
    models = {}
    
    for ba in df_train["balancing_authority_code_eia"].unique():
        df_ba = df_train[df_train["balancing_authority_code_eia"] == ba].copy()
        
        y = df_ba["demand_adjusted_mwh"].values
        X = df_ba[["hour", "dayofweek", "dayofyear", "is_holiday",
                   "load_24", "load_168", "avg_temp_K", "temp_s95", "temp_s99"]].values
        
        gam = LinearGAM(
            s(0, n_splines=24, basis="cp") +   # hour (cyclic)
            s(1, n_splines=7) +                 # dayofweek
            s(2, n_splines=10, basis="cp") +    # dayofyear (cyclic)
            f(3) +                              # is_holiday (categorical)
            s(4, n_splines=10) +                # load_24
            s(5, n_splines=10) +                # load_168
            s(6, n_splines=10) +                # avg_temp_K
            s(7, n_splines=10) +                # temp_s95
            s(8, n_splines=10) +                # temp_s99
            te(0, 6, n_splines=[8, 8])          # hour x temp interaction
        )
        
        # gridsearch is very slow at scale — use fit() with fixed lam,
        # or limit the search grid:
        gam.gridsearch(X, y, lam=np.logspace(-3, 3, 11))
        models[ba] = gam
        
        print(f"{ba}: AIC={gam.statistics_['AIC']:.1f}")
    
    return models

models = fit_gam_per_ba(df_train)



**Stack XGB with cont. update lambda**

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from scipy.optimize import minimize_scalar

features = ['hour', 'dayofweek', 'dayofyear', 'is_holiday', 'load_24', 'load_168', 'avg_temp_K', 'temp_s95', 'temp_s99']

final_preds_val = {}
ba_metrics = {}
ba_states = {}  # saves model state for test evaluation

for ba in df_train['balancing_authority_code_eia'].unique():
    df_train_ba = df_train[df_train['balancing_authority_code_eia'] == ba].copy()
    df_val_ba   = df_val[df_val['balancing_authority_code_eia'] == ba].copy()

    if len(df_train_ba) < 50 or len(df_val_ba) == 0 or ba not in models:
        continue

    # Convert to plain numpy/pandas immediately to avoid PyArrow issues
    df_train_ba[features] = df_train_ba[features].astype(float)
    df_val_ba[features]   = df_val_ba[features].astype(float)

    # 1. GAM predictions
    gam = models[ba]
    gam_pred_train = gam.predict(df_train_ba[features].to_numpy(dtype=float)).reshape(-1)
    gam_pred_val   = gam.predict(df_val_ba[features].to_numpy(dtype=float)).reshape(-1)

    y_train_actual = df_train_ba['demand_adjusted_mwh'].to_numpy(dtype=float)
    y_val_actual   = df_val_ba['demand_adjusted_mwh'].to_numpy(dtype=float)

    residuals_train = (y_train_actual - gam_pred_train).astype(float)

    # 2. Initial XGBoost fit on training residuals
    X_train_xgb = df_train_ba[features].copy()
    X_train_xgb['gam_pred'] = gam_pred_train

    gs = GridSearchCV(
        estimator=xgb.XGBRegressor(objective='reg:squarederror', random_state=42),
        param_grid={'n_estimators': [100, 200], 'max_depth': [3, 4]},
        cv=3, scoring='neg_mean_squared_error'
    )
    gs.fit(X_train_xgb, residuals_train)
    xgb_model = gs.best_estimator_

    # 3. Sliding window prediction loop
    current_X_history   = X_train_xgb.copy()
    current_res_history = list(residuals_train)

    X_val_xgb_all = df_val_ba[features].copy()
    X_val_xgb_all['gam_pred'] = gam_pred_val

    updated_preds = []
    batch_size  = 24
    window      = 2000
    tune_window = 168
    lambda_val  = 1.0

    for i in range(0, len(X_val_xgb_all), batch_size):
        X_batch        = X_val_xgb_all.iloc[i : i + batch_size]
        y_batch_actual = y_val_actual[i : i + batch_size]
        gam_batch_pred = gam_pred_val[i : i + batch_size]

        # A) Predict
        res_pred = xgb_model.predict(X_batch)
        updated_preds.extend(gam_batch_pred + (lambda_val * res_pred))

        # B) Update history
        observed_residuals = y_batch_actual - gam_batch_pred
        current_res_history.extend(observed_residuals.tolist())
        current_X_history   = pd.concat([current_X_history, X_batch]).iloc[-window:]
        current_res_history = current_res_history[-window:]

        # C) Retrain and optimize lambda
        X_train_window   = current_X_history.iloc[:-tune_window]
        res_train_window = current_res_history[:-tune_window]
        X_tune_window    = current_X_history.iloc[-tune_window:]
        res_tune_window  = np.array(current_res_history[-tune_window:], dtype=float)

        xgb_model.fit(X_train_window, res_train_window)
        xgb_preds_tune = xgb_model.predict(X_tune_window)

        def objective_function(l):
            return np.mean((res_tune_window - (l * xgb_preds_tune)) ** 2)

        opt_result = minimize_scalar(objective_function, bounds=(0.0, 1.0), method='bounded')
        lambda_val = opt_result.x

    # 4. Evaluation
    final_preds_val[ba] = np.array(updated_preds, dtype=float)
    y_true = y_val_actual[:len(final_preds_val[ba])]
    y_pred = final_preds_val[ba]

    rmse  = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape  = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-5))) * 100
    wmape = (np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)) * 100

    ba_metrics[ba] = {'RMSE': rmse, 'MAPE': mape, 'WMAPE': wmape}
    ba_states[ba] = {
        'xgb_model':         xgb_model,
        'lambda_val':        lambda_val,
        'X_history':         current_X_history.copy(),
        'res_history':       list(current_res_history),
    }
    print(f"BA {ba}: RMSE={rmse:.2f}, MAPE={mape:.2f}%, WMAPE={wmape:.2f}%")

# Summary
print(f"\n{'OVERALL PERFORMANCE':-^40}")
print(f"BAs evaluated: {len(ba_metrics)}")
print(f"Avg RMSE:  {np.mean([m['RMSE']  for m in ba_metrics.values()]):.2f}")
print(f"Avg MAPE:  {np.mean([m['MAPE']  for m in ba_metrics.values()]):.2f}%")
print(f"Avg WMAPE: {np.mean([m['WMAPE'] for m in ba_metrics.values()]):.2f}%")




In [ ]:
#  National aggregation: Simple Sum 
bas = list(final_preds_val.keys())
T   = min(len(final_preds_val[ba]) for ba in bas)

# BA-level predictions and actuals
pred_matrix = np.stack([final_preds_val[ba][:T] for ba in bas], axis=1)  # (T, n_BAs)
actual_matrix = np.stack([
    df_val[df_val["balancing_authority_code_eia"] == ba]["demand_adjusted_mwh"]
          .to_numpy(dtype=float)[:T]
    for ba in bas
], axis=1)  # (T, n_BAs)

# National totals
national_pred   = pred_matrix.sum(axis=1)
national_actual = actual_matrix.sum(axis=1)

# Metrics
def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

def rmse_score(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape_score(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-5))) * 100

print("BA-LEVEL PERFORMANCE")
print(f"Avg WMAPE: {np.mean([m['WMAPE'] for m in ba_metrics.values()]):.2f}%")
print(f"Avg RMSE:  {np.mean([m['RMSE']  for m in ba_metrics.values()]):.0f}")
print(f"Avg MAPE:  {np.mean([m['MAPE']  for m in ba_metrics.values()]):.2f}%")

print("NATIONAL LEVEL (Simple Sum")
print(f"WMAPE: {wmape(national_actual, national_pred):.2f}%")
print(f"RMSE:  {rmse_score(national_actual, national_pred):.0f}")
print(f"MAPE:  {mape_score(national_actual, national_pred):.2f}%")



In [ ]:
#Additional eval metrics
from sklearn.metrics import r2_score, mean_absolute_error
print("BA-LEVEL PERFORMANCE (Extended Metrics)")
for ba, metrics in ba_metrics.items():
    y_true = df_val[df_val["balancing_authority_code_eia"] == ba]["demand_adjusted_mwh"].to_numpy(dtype=float)[:len(final_preds_val[ba])]
    y_pred = final_preds_val[ba]

    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)

    print(f"BA {ba}: R²={r2:.4f}, MAE={mae:.2f}, RMSE={metrics['RMSE']:.2f}, MAPE={metrics['MAPE']:.2f}%, WMAPE={metrics['WMAPE']:.2f}%")
    

**Hyperparameter Tuning**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize
from pymoo.operators.sampling.rnd import FloatRandomSampling

# --- GLOBAL CHART STYLING ---
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]
plt.rcParams["font.size"] = 14 
plt.rcParams["axes.titlesize"] = 22
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16
plt.rcParams["legend.fontsize"] = 16
plt.rcParams["axes.labelweight"] = "bold"
sns.set_style("whitegrid", {'axes.grid': True, 'grid.linestyle': '--'})

#1. DEFINE THE OPTIMIZATION PROBLEM
class XGBHyperparameterProblem(ElementwiseProblem):
    def __init__(self, X_train, y_train, X_val, y_val):
        super().__init__(n_var=5,
                         n_obj=1,
                         n_constr=0,
                         xl=np.array([100, 3, 0.01, 0.5, 0.5]),
                         xu=np.array([1000, 10, 0.3, 1.0, 1.0]))
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val

    def _evaluate(self, x, out, *args, **kwargs):
        model = XGBRegressor(
            n_estimators=int(x[0]),
            max_depth=int(x[1]),
            learning_rate=x[2],
            subsample=x[3],
            colsample_bytree=x[4],
            random_state=42,
            n_jobs=-1
        )
        model.fit(self.X_train, self.y_train)
        preds = model.predict(self.X_val)
        out["F"] = np.sqrt(mean_squared_error(self.y_val, preds))

#2. RUN GENETIC ALGORITHM
problem = XGBHyperparameterProblem(X_train, y_train, X_val, y_val)
algorithm = GA(pop_size=20, sampling=FloatRandomSampling(), eliminate_duplicates=True)

print("Starting Genetic Algorithm Optimization...")
res = minimize(problem, algorithm, termination=('n_gen', 15), seed=1, verbose=True)


X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

# 2. TRAIN FINAL OPTIMIZED MODEL
best_params = res.X
final_model = XGBRegressor(
    n_estimators=int(best_params[0]),
    max_depth=int(best_params[1]),
    learning_rate=best_params[2],
    subsample=best_params[3],
    colsample_bytree=best_params[4],
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train_full, y_train_full)

# Predictions for metrics
y_pred_test = final_model.predict(X_test)
y_pred_train = final_model.predict(X_train)
y_pred_val = final_model.predict(X_val)

# 3. PERFORMANCE VISUALIZATIONS

# Metric Comparison
def get_metrics(y_true, y_pred):
    return {'R^2': r2_score(y_true, y_pred), 'MAE': mean_absolute_error(y_true, y_pred), 'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))}

m_train, m_val, m_test = get_metrics(y_train, y_pred_train), get_metrics(y_val, y_pred_val), get_metrics(y_test, y_pred_test)

fig, axes = plt.subplots(1, 3, figsize=(22, 7), dpi=150)
for i, m in enumerate(['R^2', 'MAE', 'RMSE']):
    vals = [m_train[m], m_val[m], m_test[m]]
    sns.barplot(x=['Train', 'Val', 'Test'], y=vals, ax=axes[i], palette='viridis', edgecolor='black')
    axes[i].set_title(f'Optimized {m}', fontweight='bold')
    for p in axes[i].patches:
        axes[i].annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Predicted vs Actual
plt.figure(figsize=(10, 8), dpi=150)
plt.scatter(y_test, y_pred_test, alpha=0.4, s=30, color='#440154', edgecolors='white', linewidth=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=3, label='Ideal')
plt.title('Final Model: Predicted vs. Actual Demand', fontweight='bold')
plt.xlabel('Actual Demand (MWh)')
plt.ylabel('Predicted Demand (MWh)')
plt.legend()
plt.show()

# Feature Importance
plt.figure(figsize=(12, 8), dpi=150)
feat_importances = pd.Series(final_model.feature_importances_, index=X_train.columns)
feat_importances.nlargest(10).sort_values().plot(kind='barh', color=sns.color_palette("magma", 10), edgecolor='black')
plt.title('Top 10 Feature Importances (GA-Optimized XGBoost)', fontweight='bold')
plt.xlabel('Importance')
plt.show()

# Final Metrics Print
print("\n" + "="*50)
print(f"GA OPTIMIZED PARAMS: n={int(best_params[0])}, depth={int(best_params[1])}, lr={best_params[2]:.4f}")
print(f"TEST R^2: {m_test['R^2']:.4f} | TEST MAE: {m_test['MAE']:.2f}")
print("="*50)